# AML Synthetic Data Generator -- Complete Pipeline
---
**Single notebook** that generates all master data tables and a fully-labeled transaction
dataset with **all 92 schema columns** and embedded AML typologies.

### Output Files
| File | Description | Primary Key |
|------|-------------|-------------|
| `customers.csv` | Individual + Entity customer profiles | `customer_cif` |
| `accounts.csv` | Bank accounts linked to customers | `account_number` |
| `wallets.csv` | PPI wallets linked to customers | `wallet_id` |
| `devices.csv` | Device fingerprints per customer | `device_id` |
| `transactions_final.csv` | All transactions with 92 columns + AML labels | `transaction_id` |

### AML Typologies (injected during generation)
| # | Typology | Pattern |
|---|----------|--------|
| 1 | Structuring (Smurfing) | Sub-threshold cash deposits then consolidation |
| 2 | Circular Transaction Loops | A -> B -> C -> A ring transfers |
| 3 | Funnel Account Networks | Many-to-one then rapid outflow |
| 4 | Pass-Through Transit Hubs | Receive large sum, forward within minutes |
| 5 | Rapid Multi-Hop Layering | 8-10 hop chains within hours |


## 1 -- Environment Setup


In [ ]:
import random, string, hashlib, uuid, json, os, math
from datetime import datetime, timedelta, date
from collections import defaultdict, Counter
import csv
import warnings
warnings.filterwarnings('ignore')

random.seed(42)

OUTPUT_DIR = 'aml_synthetic_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {os.path.abspath(OUTPUT_DIR)}')



## 2 -- Configuration (includes Fraud % and Typology Weights)


In [ ]:
CONFIG = {
    # ── Volume ──
    "num_customers_individual": 4000,
    "num_customers_entity": 1000,
    "num_accounts_per_customer_range": [1, 3],
    "num_wallets_fraction": 0.30,
    "num_transactions_per_account_range": [10, 120],
    "date_range_start": "2024-01-01",
    "date_range_end": "2025-03-31",

    # ── Fraud / Typology Configuration ──
    "target_fraud_pct": 0.10,   # 10% of final transactions will be AML-labeled
    "typology_weights": {
        "Structuring (Smurfing)":    0.25,
        "Circular Transaction Loop": 0.15,
        "Funnel Account Network":    0.30,
        "Pass-Through Transit Hub":  0.10,
        "Rapid Multi-Hop Layering":  0.20,
    },
    "avg_txns_per_scenario": {
        "Structuring (Smurfing)":    9,
        "Circular Transaction Loop": 8,
        "Funnel Account Network":    35,
        "Pass-Through Transit Hub":  2,
        "Rapid Multi-Hop Layering":  9,
    },

    # ── Currency ──
    "primary_currency": "INR",
    "fx_currencies": ["USD", "GBP", "EUR", "AED", "SGD"],
    "fx_probability": 0.03,

    # ── Risk distribution ──
    "risk_weights": {"Low": 0.60, "Medium": 0.30, "High": 0.10},

    # ── PEP / HNI / Minor ──
    "pep_probability": 0.02,
    "hni_probability": 0.05,
    "minor_probability": 0.03,

    # ── Income bands (INR) ──
    "income_bands": {
        "Low":    [100000, 500000],
        "Medium": [500001, 2500000],
        "High":   [2500001, 10000000],
        "HNI":    [10000001, 100000000]
    },

    # ── Wallet limits (RBI PPI norms) ──
    "wallet_kyc_categories": {
        "Minimum KYC": {
            "per_txn": 10000, "daily": 10000, "monthly": 10000,
            "annual": 120000, "max_balance": 10000
        },
        "Full KYC": {
            "per_txn": 200000, "daily": 200000, "monthly": 200000,
            "annual": 2400000, "max_balance": 200000
        }
    },

    # ── IFSC pool ──
    "bank_ifsc_prefixes": [
        "SBIN0", "HDFC0", "ICIC0", "UTIB0", "PUNB0",
        "BARB0", "CNRB0", "KKBK0", "IOBA0", "BKID0"
    ],

    # ── MCC codes (ISO 18245) ──
    "mcc_codes": {
        "5411": "Grocery Stores", "5541": "Service Stations",
        "5812": "Restaurants", "5912": "Drug Stores",
        "4814": "Telecom Services", "6012": "Financial Institutions",
        "7011": "Hotels/Motels", "5691": "Clothing Stores",
        "5045": "Computers/Peripherals", "8062": "Hospitals",
        "4899": "Cable/Utilities", "5999": "Miscellaneous Retail",
        "6051": "Quasi Cash - Money Orders", "6211": "Security Brokers/Dealers",
        "7995": "Gambling/Betting"
    },

    # ── Transaction channels ──
    "bank_channels": [
        "NEFT", "RTGS", "IMPS", "UPI", "Branch Cash",
        "ATM", "Cheque", "Internet Banking", "Mobile Banking",
        "POS", "Demand Draft"
    ],
    "ppi_channels": ["UPI", "QR Scan", "In-App Transfer", "NFC Tap", "Online"],

    # ── Device / Auth ──
    "auth_methods": ["OTP", "PIN", "Biometric", "MPIN", "Password"],
    "browsers": [
        "Chrome/124.0", "Safari/17.4", "Firefox/125.0",
        "Edge/124.0", "SamsungBrowser/24.0"
    ],
    "app_versions": ["v8.1.2", "v8.2.0", "v9.0.1", "v9.1.0", "v7.5.3"],

    # ── Geography ──
    "indian_states": [
        "Maharashtra", "Delhi", "Karnataka", "Tamil Nadu", "Telangana",
        "Gujarat", "Rajasthan", "Uttar Pradesh", "West Bengal", "Kerala",
        "Madhya Pradesh", "Punjab", "Haryana", "Bihar", "Odisha"
    ],
    "cities_by_state": {
        "Maharashtra":   ["Mumbai", "Pune", "Nagpur", "Nashik"],
        "Delhi":         ["New Delhi", "Dwarka", "Rohini", "Saket"],
        "Karnataka":     ["Bengaluru", "Mysuru", "Hubli", "Mangaluru"],
        "Tamil Nadu":    ["Chennai", "Coimbatore", "Madurai", "Salem"],
        "Telangana":     ["Hyderabad", "Warangal", "Nizamabad", "Karimnagar"],
        "Gujarat":       ["Ahmedabad", "Surat", "Vadodara", "Rajkot"],
        "Rajasthan":     ["Jaipur", "Jodhpur", "Udaipur", "Kota"],
        "Uttar Pradesh": ["Lucknow", "Noida", "Agra", "Varanasi"],
        "West Bengal":   ["Kolkata", "Howrah", "Siliguri", "Durgapur"],
        "Kerala":        ["Kochi", "Thiruvananthapuram", "Kozhikode", "Thrissur"],
        "Madhya Pradesh":["Bhopal", "Indore", "Gwalior", "Jabalpur"],
        "Punjab":        ["Chandigarh", "Ludhiana", "Amritsar", "Jalandhar"],
        "Haryana":       ["Gurugram", "Faridabad", "Karnal", "Panipat"],
        "Bihar":         ["Patna", "Gaya", "Muzaffarpur", "Bhagalpur"],
        "Odisha":        ["Bhubaneswar", "Cuttack", "Rourkela", "Puri"]
    },

    # ── Occupations ──
    "occupations_individual": [
        "Salaried - IT", "Salaried - Banking", "Salaried - Government",
        "Self-Employed - Retail", "Self-Employed - Professional",
        "Business Owner", "Freelancer", "Student", "Retired",
        "Agriculture", "Doctor", "Lawyer", "Consultant"
    ],
    "entity_types": [
        "Private Limited", "Public Limited", "LLP",
        "Partnership", "Trust", "Society", "Sole Proprietorship",
        "HUF", "Government Body"
    ],
    "entity_industries": [
        "IT Services", "Manufacturing", "Trading", "Real Estate",
        "Financial Services", "Healthcare", "Education", "Hospitality",
        "Logistics", "Textiles", "Pharmaceuticals", "Agriculture",
        "Construction", "Retail", "Import/Export"
    ]
}

with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)
print("Configuration saved (including fraud % and typology weights)")



## 3 -- Full 92-Column Schema Reference
Every transaction row will contain exactly these 92 fields plus 3 AML label fields (95 total).


In [ ]:
# Definitive ordered list of ALL 92 schema columns + 3 label columns = 95 total
TRANSACTION_COLUMNS = [
    # -- Transaction Identifiers (SN 1-22) --
    "transaction_id",                    # SN1  - Unique transaction reference
    "timestamp",                         # SN2  - HH:MM:SS
    "datestamp",                          # SN3  - DD-MM-YYYY
    "transaction_amount",                # SN4  - Monetary value
    "currency",                          # SN5  - ISO currency code
    "transaction_type_dr_cr",            # SN6  - Debit (Dr) / Credit (Cr)
    "transaction_mode_channel_bank",     # SN7  - Bank channel (NEFT/RTGS/UPI etc)
    "cash_flag",                         # SN8  - Y/N physical cash
    "transaction_type_ppi",              # SN9  - PPI transaction type
    "transaction_mode_channel_ppi",      # SN10 - PPI digital channel
    "transaction_status",                # SN11 - Success/Failed/Pending/Reversed
    "wallet_balance_before",             # SN12 - PPI balance before
    "wallet_balance_after",              # SN13 - PPI balance after
    "source_of_funds_wallet",            # SN14 - Wallet load source
    "load_instrument_type",              # SN15 - Payment method for wallet load
    "load_source_account_card_details",  # SN16 - Masked source account/card
    "beneficiary_wallet_id_vpa",         # SN17 - Recipient wallet/UPI VPA
    "merchant_id",                       # SN18 - Merchant identifier
    "merchant_name",                     # SN19 - Business name
    "merchant_category_code",            # SN20 - MCC (ISO 18245)
    "merchant_location",                 # SN21 - Merchant location
    "refund_chargeback_flag",            # SN22 - Y/N refund/chargeback

    # -- Participant Information (SN 23-38) --
    "customer_account_number",           # SN23 - Customer bank account
    "account_wallet_status",             # SN24 - Active/Frozen/Dormant etc
    "non_face_to_face_flag",             # SN25 - Y/N remote onboarding
    "pep_flag",                          # SN26 - Politically Exposed Person
    "hni_flag",                          # SN27 - High Net Worth Individual
    "minor_flag",                        # SN28 - Under 18 flag
    "customer_branch_ifsc_code",         # SN29 - Home branch IFSC
    "customer_cif_id",                   # SN30 - CIF number
    "customer_cif_creation_date",        # SN31 - CIF creation date
    "annual_income",                     # SN32 - Annual income (INR)
    "counterparty_account_number",       # SN33 - Beneficiary/sender account
    "counterparty_branch_ifsc_swift",    # SN34 - Counterparty IFSC/SWIFT
    "customer_name",                     # SN35 - Full legal name
    "counterparty_name",                 # SN36 - Counterparty name
    "sender_country_code",               # SN37 - Sender ISO country
    "receiver_country_code",             # SN38 - Receiver ISO country

    # -- Customer Profile Data (SN 39-74) --
    "customer_current_risk_score",       # SN39 - Low/Medium/High
    "customer_type",                     # SN40 - Individual/Non-Individual
    "customer_entity_type",              # SN41 - Detailed entity type
    "account_category",                  # SN42 - Savings/Current/NRE etc
    "account_type",                      # SN43 - Regular/Premium/Basic etc
    "account_wallet_opening_date",       # SN44 - Account/wallet open date
    "customer_occupation_industry",      # SN45 - Occupation or industry
    "vkyc_flag",                         # SN46 - Video KYC flag
    "kyc_update_date",                   # SN47 - Last KYC update
    "account_wallet_inoperative_date",   # SN48 - Inoperative status date
    "source_of_funds",                   # SN49 - Wealth/income origin
    "tax_residency",                     # SN50 - India/Other
    "nationality",                       # SN51 - Country of citizenship
    "citizenship",                       # SN52 - Legal citizenship
    "residency",                         # SN53 - Resident/NRI
    "date_of_incorporation",             # SN54 - Entity incorporation date
    "place_of_incorporation",            # SN55 - Entity registration location
    "beneficial_owner_types",            # SN56 - UBO classification
    "passive_nfe",                       # SN57 - Passive Non-Financial Entity
    "address_registered_office",         # SN58 - Entity registered office
    "address_place_of_business",         # SN59 - Entity business address
    "address_beneficial_owners",         # SN60 - UBO addresses
    "address_individual_customer",       # SN61 - Individual residential address
    "date_of_birth",                     # SN62 - Individual DOB
    "father_spouse_name",                # SN63 - Father/spouse name
    "identification_proof_doc_no",       # SN64 - OVD document number
    "entity_identification_proof_doc_no",# SN65 - Entity registration number
    "credit_summation_period",           # SN66 - Total credits in period
    "debit_summation_period",            # SN67 - Total debits in period
    "professional_experience_years",     # SN68 - Work experience
    "cif_beneficial_owners",             # SN69 - UBO customer IDs
    "name_beneficial_owners",            # SN70 - UBO names
    "mobile_number",                     # SN71 - Registered mobile
    "pan",                               # SN72 - PAN number
    "aadhaar_number",                    # SN73 - Aadhaar (masked)
    "email_id",                          # SN74 - Email address

    # -- Wallet Account Data (SN 75-82) --
    "wallet_kyc_category",               # SN75 - Min KYC / Full KYC
    "wallet_account_id",                 # SN76 - Wallet ID
    "escrow_account_linked",             # SN77 - Escrow account
    "transaction_limit_per_txn",         # SN78 - Per-txn limit
    "daily_transaction_limit",           # SN79 - Daily limit
    "monthly_transaction_limit",         # SN80 - Monthly limit
    "annual_transaction_limit",          # SN81 - Annual limit
    "maximum_wallet_balance_limit",      # SN82 - Max wallet balance

    # -- Device & Channel Data (SN 83-92) --
    "device_id_fingerprint",             # SN83 - Device ID/fingerprint
    "ip_address",                        # SN84 - Originating IP
    "geo_location_city_country",         # SN85 - Geo from IP/GPS
    "gps_coordinates_lat",               # SN86 - GPS latitude
    "gps_coordinates_lon",               # SN86 - GPS longitude (split)
    "browser_app_information",           # SN87 - Browser/app version
    "session_id",                        # SN88 - Session identifier
    "authentication_method",             # SN89 - OTP/PIN/Biometric
    "vpn_flag",                          # SN90 - VPN/proxy/Tor flag
    "emulator_flag",                     # SN91 - Emulator flag
    "customer_address_lat",              # SN92 - Registered address lat
    "customer_address_lon",              # SN92 - Registered address lon (split)

    # -- AML Labels (3 additional) --
    "is_aml",                            # 0/1 label
    "aml_typology",                      # Typology name or empty
    "typology_group_id",                 # Scenario group ID or empty
]

print(f"Total columns defined: {len(TRANSACTION_COLUMNS)}")
print(f"  Schema columns (SN 1-92): {len(TRANSACTION_COLUMNS) - 3}")
print(f"  AML label columns: 3")



## 4 -- Helper Functions


In [ ]:
# -- Realistic ID generators --

def generate_cif():
    return str(random.randint(100000000000, 999999999999))

def generate_account_number(ifsc_prefix):
    bank = ifsc_prefix[:4]
    if bank == "SBIN":
        return str(random.randint(10000000000, 99999999999))
    elif bank == "HDFC":
        return str(random.randint(10000000000000, 99999999999999))
    elif bank == "ICIC":
        return str(random.randint(100000000000, 999999999999))
    elif bank == "UTIB":
        return str(random.randint(9100000000000, 9299999999999))
    elif bank == "PUNB":
        return str(random.randint(1000000000000000, 9999999999999999))
    else:
        length = random.choice([11, 12, 13, 14])
        return str(random.randint(10**(length-1), 10**length - 1))

def generate_ifsc():
    prefix = random.choice(CONFIG["bank_ifsc_prefixes"])
    return prefix + str(random.randint(10000, 99999))

def generate_pan(is_entity=False):
    first3 = ''.join(random.choices(string.ascii_uppercase, k=3))
    fourth = random.choice(["C","H","F","A","T","B","L","J","G"]) if is_entity else "P"
    fifth = random.choice(string.ascii_uppercase)
    digits = ''.join(random.choices(string.digits, k=4))
    last = random.choice(string.ascii_uppercase)
    return first3 + fourth + fifth + digits + last

def generate_aadhaar():
    return str(random.randint(2, 9)) + ''.join(random.choices(string.digits, k=11))

def generate_aadhaar_masked():
    return "XXXX-XXXX-" + ''.join(random.choices(string.digits, k=4))

def generate_mobile():
    return str(random.choice([6,7,8,9])) + ''.join(random.choices(string.digits, k=9))

def generate_email(name):
    domains = ["gmail.com", "yahoo.co.in", "outlook.com", "rediffmail.com", "hotmail.com"]
    clean = name.lower().replace(" ", ".").replace("'", "")
    return f"{clean}{random.randint(1,999)}@{random.choice(domains)}"

def generate_vpa(name):
    handles = ["oksbi", "okaxis", "okicici", "okhdfcbank", "ybl", "paytm", "ibl", "upi"]
    clean = name.lower().replace(" ", "")[:10]
    return f"{clean}{random.randint(1,99)}@{random.choice(handles)}"

def generate_wallet_id():
    return "WLT" + ''.join(random.choices(string.digits, k=12))

def generate_merchant_id():
    return "MID" + ''.join(random.choices(string.ascii_uppercase + string.digits, k=12))

def generate_device_id():
    return hashlib.sha256(uuid.uuid4().bytes).hexdigest()[:32]

def generate_ip():
    first_octet = random.choice([49, 59, 103, 106, 117, 122, 157, 182, 203])
    return f"{first_octet}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

def generate_gps(city, state):
    coords = {
        "Mumbai": (19.076, 72.878), "Pune": (18.520, 73.856),
        "New Delhi": (28.614, 77.209), "Bengaluru": (12.972, 77.594),
        "Chennai": (13.083, 80.271), "Hyderabad": (17.385, 78.487),
        "Ahmedabad": (23.023, 72.571), "Kolkata": (22.573, 88.364),
        "Jaipur": (26.912, 75.787), "Lucknow": (26.847, 80.947),
        "Kochi": (9.932, 76.267), "Bhopal": (23.260, 77.413),
        "Chandigarh": (30.734, 76.779), "Patna": (25.611, 85.144),
        "Bhubaneswar": (20.297, 85.825),
    }
    base = coords.get(city, (20.5 + random.uniform(-5, 5), 78.0 + random.uniform(-5, 5)))
    return (round(base[0] + random.uniform(-0.05, 0.05), 6),
            round(base[1] + random.uniform(-0.05, 0.05), 6))

def random_date(start_str, end_str):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end = datetime.strptime(end_str, "%Y-%m-%d")
    delta = (end - start).days
    if delta <= 0:
        return start
    return start + timedelta(days=random.randint(0, delta))

def weighted_choice(weight_dict):
    items = list(weight_dict.keys())
    weights = list(weight_dict.values())
    return random.choices(items, weights=weights, k=1)[0]

def generate_txn_id():
    return "TXN" + ''.join(random.choices(string.ascii_uppercase + string.digits, k=16))

# -- Name generators --
FIRST_NAMES_M = ["Rahul","Amit","Suresh","Vikram","Arjun","Rohan","Kiran","Deepak",
    "Manoj","Rajesh","Sanjay","Anil","Pradeep","Nikhil","Varun",
    "Aditya","Akash","Harsh","Yash","Gaurav","Sachin","Pranav",
    "Ankit","Ravi","Sandeep","Mohit","Tushar","Vivek","Ashish","Naveen"]
FIRST_NAMES_F = ["Priya","Anita","Sunita","Kavita","Neha","Pooja","Swati","Meera",
    "Divya","Sneha","Ritu","Pallavi","Shruti","Aditi","Anjali",
    "Nisha","Rekha","Sonal","Tanvi","Bhavna","Komal","Jyoti",
    "Asha","Geeta","Lata","Mamta","Rashmi","Sapna","Shweta","Vandana"]
LAST_NAMES = ["Sharma","Verma","Patel","Gupta","Singh","Kumar","Reddy","Nair",
    "Iyer","Joshi","Deshmukh","Rao","Pillai","Mehta","Shah",
    "Agarwal","Mishra","Bhat","Yadav","Chauhan","Tiwari","Pandey",
    "Malhotra","Kapoor","Bose","Chatterjee","Mukherjee","Das","Banerjee","Sen"]
ENTITY_PREFIXES = ["Shri","Om","Bharat","National","Premier","Golden","Silver",
    "Diamond","Royal","Star","Global","United","Pioneer","Excel","Apex"]
ENTITY_SUFFIXES = ["Enterprises","Trading Co","Industries","Solutions","Services",
    "Exports","Imports","Associates","Group","Holdings",
    "Infra","Ventures","Technologies","Logistics","Capital"]
MERCHANT_NAMES = [
    "Reliance Fresh","Big Bazaar","DMart","More Supermarket",
    "Swiggy","Zomato","Amazon India","Flipkart","Myntra",
    "BookMyShow","MakeMyTrip","IRCTC","Uber India","Ola Cabs",
    "Jio Payments","Airtel Store","Apollo Pharmacy","MedPlus",
    "Croma Electronics","Vijay Sales","Tanishq","Titan Eye+",
    "PVR Cinemas","Decathlon India","Nykaa","Lenskart",
    "Urban Company","PhonePe Merchant","Google Pay Merchant","Paytm Mall"]

def random_individual_name():
    if random.random() < 0.5:
        return random.choice(FIRST_NAMES_M) + " " + random.choice(LAST_NAMES)
    return random.choice(FIRST_NAMES_F) + " " + random.choice(LAST_NAMES)

def random_entity_name():
    return random.choice(ENTITY_PREFIXES) + " " + random.choice(ENTITY_SUFFIXES)

def random_father_name():
    return random.choice(FIRST_NAMES_M) + " " + random.choice(LAST_NAMES)

print("Helpers loaded")



## 5 -- Generate Customer Master Data


In [ ]:
customers = []
cif_set = set()

def make_individual_customer():
    cif = generate_cif()
    while cif in cif_set:
        cif = generate_cif()
    cif_set.add(cif)

    state = random.choice(CONFIG["indian_states"])
    city = random.choice(CONFIG["cities_by_state"][state])
    gps = generate_gps(city, state)
    name = random_individual_name()

    is_pep = random.random() < CONFIG["pep_probability"]
    is_hni = random.random() < CONFIG["hni_probability"]
    is_minor = random.random() < CONFIG["minor_probability"]
    risk = weighted_choice(CONFIG["risk_weights"])
    if is_pep:
        risk = "High"

    if is_hni:
        income_band = CONFIG["income_bands"]["HNI"]
    elif risk == "High":
        income_band = CONFIG["income_bands"][random.choice(["Medium", "High"])]
    else:
        income_band = CONFIG["income_bands"][risk]
    annual_income = random.randint(*income_band)

    dob = random_date("1960-01-01", "2006-12-31") if not is_minor else random_date("2008-01-01", "2015-12-31")
    cif_creation = random_date("2015-01-01", "2024-06-30")
    kyc_date = random_date(cif_creation.strftime("%Y-%m-%d"), "2025-03-15")

    return {
        "customer_cif": cif, "customer_name": name,
        "customer_type": "Individual", "customer_entity_type": "Individual",
        "date_of_birth": dob.strftime("%d-%m-%Y"),
        "father_spouse_name": random_father_name(),
        "nationality": "Indian", "citizenship": "Indian",
        "residency": random.choices(["Resident", "NRI"], weights=[0.95, 0.05])[0],
        "tax_residency": random.choices(["India", "Other"], weights=[0.96, 0.04])[0],
        "pan": generate_pan(False), "aadhaar": generate_aadhaar(),
        "aadhaar_masked": generate_aadhaar_masked(),
        "identification_doc_no": "AADHAAR-" + generate_aadhaar()[:4] + "XXXX" + generate_aadhaar()[-4:],
        "mobile_number": generate_mobile(), "email_id": generate_email(name),
        "address_individual": f"{random.randint(1,500)}, {random.choice(['MG Road','Station Road','NH Highway','Park Street','Ring Road','Gandhi Nagar','Nehru Place'])}, {city}, {state} - {random.randint(100000,999999)}",
        "state": state, "city": city,
        "address_lat": gps[0], "address_lon": gps[1],
        "occupation_industry": random.choice(CONFIG["occupations_individual"]),
        "annual_income": annual_income,
        "professional_experience_years": 0 if is_minor else random.randint(0, 40),
        "source_of_funds": random.choice(["Salary","Business Income","Investment Returns","Rental Income","Family Support","Pension","Agriculture Income"]),
        "customer_risk_score": risk,
        "pep_flag": "Y" if is_pep else "N",
        "hni_flag": "Y" if is_hni else "N",
        "minor_flag": "Y" if is_minor else "N",
        "non_face_to_face_flag": random.choices(["Y","N"], weights=[0.35,0.65])[0],
        "vkyc_flag": random.choices(["Y","N"], weights=[0.25,0.75])[0],
        "cif_creation_date": cif_creation.strftime("%d-%m-%Y"),
        "kyc_update_date": kyc_date.strftime("%d-%m-%Y"),
        "date_of_incorporation": "", "place_of_incorporation": "",
        "beneficial_owner_types": "", "passive_nfe": "",
        "address_registered_office": "", "address_place_of_business": "",
        "address_beneficial_owners": "", "entity_identification_doc_no": "",
        "cif_beneficial_owners": "", "name_beneficial_owners": "",
    }

def make_entity_customer():
    cif = generate_cif()
    while cif in cif_set:
        cif = generate_cif()
    cif_set.add(cif)

    state = random.choice(CONFIG["indian_states"])
    city = random.choice(CONFIG["cities_by_state"][state])
    gps = generate_gps(city, state)
    entity_name = random_entity_name()
    entity_type = random.choice(CONFIG["entity_types"])
    risk = weighted_choice(CONFIG["risk_weights"])
    inc_date = random_date("1990-01-01", "2024-01-01")
    cif_creation = random_date(inc_date.strftime("%Y-%m-%d"), "2024-06-30")
    if cif_creation < inc_date:
        cif_creation = inc_date + timedelta(days=random.randint(1, 365))
    kyc_date = random_date(cif_creation.strftime("%Y-%m-%d"), "2025-03-15")
    ubo_name = random_individual_name()

    return {
        "customer_cif": cif, "customer_name": entity_name,
        "customer_type": "Non-Individual", "customer_entity_type": entity_type,
        "date_of_birth": "", "father_spouse_name": "",
        "nationality": "Indian", "citizenship": "",
        "residency": "Resident", "tax_residency": "India",
        "pan": generate_pan(True), "aadhaar": "", "aadhaar_masked": "",
        "identification_doc_no": "", "mobile_number": generate_mobile(),
        "email_id": generate_email(entity_name.split()[0]),
        "address_individual": "", "state": state, "city": city,
        "address_lat": gps[0], "address_lon": gps[1],
        "occupation_industry": random.choice(CONFIG["entity_industries"]),
        "annual_income": random.randint(1000000, 500000000),
        "professional_experience_years": "",
        "source_of_funds": random.choice(["Business Revenue","Investment","Government Grant","Donations","Trading Profits"]),
        "customer_risk_score": risk,
        "pep_flag": "N", "hni_flag": "N", "minor_flag": "N",
        "non_face_to_face_flag": random.choices(["Y","N"], weights=[0.20,0.80])[0],
        "vkyc_flag": "N",
        "cif_creation_date": cif_creation.strftime("%d-%m-%Y"),
        "kyc_update_date": kyc_date.strftime("%d-%m-%Y"),
        "date_of_incorporation": inc_date.strftime("%d-%m-%Y"),
        "place_of_incorporation": f"{city}, {state}",
        "beneficial_owner_types": random.choice(["Shareholding > 25%","Control through other means","Senior Managing Official"]),
        "passive_nfe": random.choices(["Y","N"], weights=[0.15,0.85])[0],
        "address_registered_office": f"{random.randint(1,200)} Corporate Park, {city}, {state} - {random.randint(100000,999999)}",
        "address_place_of_business": f"Plot {random.randint(1,500)}, Industrial Area, {city}, {state}",
        "address_beneficial_owners": f"{random.randint(1,999)}, Residential Colony, {city}, {state}",
        "entity_identification_doc_no": f"U{random.randint(10000,99999)}{random.choice(string.ascii_uppercase)}{random.choice(string.ascii_uppercase)}{random.randint(1990,2024)}PTC{random.randint(100000,999999)}",
        "cif_beneficial_owners": generate_cif(),
        "name_beneficial_owners": ubo_name,
    }

print("Generating customers...")
for _ in range(CONFIG["num_customers_individual"]):
    customers.append(make_individual_customer())
for _ in range(CONFIG["num_customers_entity"]):
    customers.append(make_entity_customer())
random.shuffle(customers)
cust_lookup = {c["customer_cif"]: c for c in customers}
print(f"Total customers: {len(customers):,}")



## 6 -- Generate Accounts


In [ ]:
accounts = []
acct_number_set = set()
cif_to_accounts = defaultdict(list)

ACCOUNT_CATEGORIES = ["Savings","Current","Salary","NRE","NRO","Fixed Deposit","Recurring Deposit"]
ACCOUNT_TYPES = ["Regular","Premium","Basic","Corporate","BSBD"]

for cust in customers:
    cif = cust["customer_cif"]
    n_accts = random.randint(*CONFIG["num_accounts_per_customer_range"])
    ifsc = generate_ifsc()

    for j in range(n_accts):
        acct_num = generate_account_number(ifsc[:5])
        while acct_num in acct_number_set:
            acct_num = generate_account_number(ifsc[:5])
        acct_number_set.add(acct_num)

        cif_dt = datetime.strptime(cust["cif_creation_date"], "%d-%m-%Y")
        start_bound = max(datetime.strptime("2015-01-01", "%Y-%m-%d"), cif_dt)
        open_date = random_date(start_bound.strftime("%Y-%m-%d"), "2024-12-31")

        is_dormant = random.random() < 0.05
        status = "Dormant" if is_dormant else random.choices(["Active","Frozen","Closed"], weights=[0.90,0.03,0.02])[0]
        inop_date = random_date(open_date.strftime("%Y-%m-%d"), "2025-03-15").strftime("%d-%m-%Y") if is_dormant else ""

        cat = random.choice(["Current","Fixed Deposit"]) if cust["customer_type"] == "Non-Individual" else random.choice(ACCOUNT_CATEGORIES)

        acct = {
            "account_number": acct_num, "customer_cif": cif,
            "customer_branch_ifsc": ifsc, "account_category": cat,
            "account_type": random.choice(ACCOUNT_TYPES), "account_status": status,
            "account_opening_date": open_date.strftime("%d-%m-%Y"),
            "inoperative_status_date": inop_date,
            "credit_summation_period": 0, "debit_summation_period": 0,
        }
        accounts.append(acct)
        cif_to_accounts[cif].append(acct_num)

acct_lookup = {a["account_number"]: a for a in accounts}
print(f"Total accounts: {len(accounts):,}")



## 7 -- Generate Wallets


In [ ]:
wallets = []
wallet_id_set = set()
cif_to_wallet = {}
wallet_lookup = {}

individual_cifs = [c["customer_cif"] for c in customers if c["customer_type"] == "Individual"]
wallet_cifs = random.sample(individual_cifs, int(len(individual_cifs) * CONFIG["num_wallets_fraction"]))

for cif in wallet_cifs:
    wid = generate_wallet_id()
    while wid in wallet_id_set:
        wid = generate_wallet_id()
    wallet_id_set.add(wid)

    kyc_cat = random.choices(["Minimum KYC","Full KYC"], weights=[0.40,0.60])[0]
    limits = CONFIG["wallet_kyc_categories"][kyc_cat]
    linked_accts = cif_to_accounts.get(cif, [])
    linked_acct = linked_accts[0] if linked_accts else ""
    cust = cust_lookup[cif]
    cif_year = cust["cif_creation_date"].split("-")[-1]
    open_date = random_date(f"{cif_year}-01-01", "2025-01-31")

    w = {
        "wallet_id": wid, "customer_cif": cif, "wallet_kyc_category": kyc_cat,
        "wallet_status": random.choices(["Active","Suspended","Closed"], weights=[0.90,0.05,0.05])[0],
        "wallet_opening_date": open_date.strftime("%d-%m-%Y"),
        "escrow_account_linked": f"ESC{random.randint(10**12, 10**13-1)}",
        "per_txn_limit": limits["per_txn"], "daily_txn_limit": limits["daily"],
        "monthly_txn_limit": limits["monthly"], "annual_txn_limit": limits["annual"],
        "max_balance_limit": limits["max_balance"], "linked_bank_account": linked_acct,
    }
    wallets.append(w)
    cif_to_wallet[cif] = wid
    wallet_lookup[wid] = w

print(f"Total wallets: {len(wallets):,}")



## 8 -- Generate Device Profiles


In [ ]:
devices = []
cif_to_devices = defaultdict(list)

for cust in customers:
    cif = cust["customer_cif"]
    n_devices = random.choices([1,2,3], weights=[0.60,0.30,0.10])[0]
    for _ in range(n_devices):
        dev = {
            "customer_cif": cif,
            "device_id": generate_device_id(), "ip_address": generate_ip(),
            "geo_city": cust["city"], "geo_country": "IN",
            "gps_lat": round(cust["address_lat"] + random.uniform(-0.01, 0.01), 6),
            "gps_lon": round(cust["address_lon"] + random.uniform(-0.01, 0.01), 6),
            "browser_app_info": random.choice(CONFIG["browsers"] + CONFIG["app_versions"]),
            "vpn_flag": random.choices(["Y","N"], weights=[0.03,0.97])[0],
            "emulator_flag": random.choices(["Y","N"], weights=[0.01,0.99])[0],
        }
        devices.append(dev)
        cif_to_devices[cif].append(dev)

print(f"Total device profiles: {len(devices):,}")



## 9 -- Transaction Builder (Full 95 Columns)


In [ ]:
used_timestamps = defaultdict(set)
txn_id_set = set()

def unique_timestamp(account_number, txn_date, preferred_hour=None):
    for _ in range(86400):
        h = preferred_hour if preferred_hour is not None else random.randint(6, 23)
        m = random.randint(0, 59)
        s = random.randint(0, 59)
        ts = f"{h:02d}:{m:02d}:{s:02d}"
        key = (txn_date, ts)
        if key not in used_timestamps[account_number]:
            used_timestamps[account_number].add(key)
            return ts
    return f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}"

active_accounts_list = [a for a in accounts if a["account_status"] in ("Active","Dormant")]
all_acct_numbers = [a["account_number"] for a in active_accounts_list]
active_only_numbers = [a["account_number"] for a in accounts if a["account_status"] == "Active"]

def build_full_txn(acct_num, cp_acct, amount, txn_date_str, txn_time,
                   direction, channel, is_aml=0, aml_typology="", group_id="",
                   is_ppi=False, ppi_txn_type="", ppi_channel=""):
    # Resolve all entities
    acct_info = acct_lookup.get(acct_num, {})
    cif = acct_info.get("customer_cif", "")
    cust = cust_lookup.get(cif, {})
    cp_info = acct_lookup.get(cp_acct, {})
    cp_cif = cp_info.get("customer_cif", "")
    cp_cust = cust_lookup.get(cp_cif, {})

    # Device
    dev_list = cif_to_devices.get(cif, [])
    dev = random.choice(dev_list) if dev_list else None

    # Wallet
    wallet_id = cif_to_wallet.get(cif, "")
    w_info = wallet_lookup.get(wallet_id, {}) if is_ppi and wallet_id else {}

    # Cash flag
    is_cash = "Y" if channel in ("Branch Cash", "ATM") else "N"

    # Merchant
    has_merchant = channel in ("POS","UPI") or (is_ppi and ppi_txn_type == "P2M") or random.random() < 0.2
    mcc = random.choice(list(CONFIG["mcc_codes"].keys())) if has_merchant else ""
    m_name = random.choice(MERCHANT_NAMES) if has_merchant else ""
    m_id = generate_merchant_id() if has_merchant else ""
    m_loc = f"{cust.get('city','')}, {cust.get('state','')}" if has_merchant else ""

    # Wallet balances
    w_before = round(random.uniform(0, float(w_info.get("max_balance_limit", 200000))), 2) if is_ppi else ""
    if is_ppi and isinstance(w_before, float):
        w_after = round(max(0, w_before - amount), 2) if direction == "Dr" else round(w_before + amount, 2)
    else:
        w_after = ""

    # FX
    is_fx = random.random() < CONFIG["fx_probability"] if not is_aml else False
    currency = random.choice(CONFIG["fx_currencies"]) if is_fx else CONFIG["primary_currency"]
    sender_cc, receiver_cc = "IN", "IN"
    if is_fx:
        if direction == "Cr":
            sender_cc = random.choice(["US","GB","AE","SG","DE"])
        else:
            receiver_cc = random.choice(["US","GB","AE","SG","DE"])

    status = random.choices(["Success","Failed","Pending","Reversed"], weights=[0.92,0.04,0.02,0.02])[0] if not is_aml else "Success"
    session_id = hashlib.md5(f"{acct_num}{txn_date_str}{random.random()}".encode()).hexdigest()[:24]

    cp_name = cp_cust.get("customer_name", "") if cp_cust else random_individual_name()

    txn = {
        # -- Transaction Identifiers (SN 1-22) --
        "transaction_id": "",  # set externally
        "timestamp": txn_time,
        "datestamp": txn_date_str,
        "transaction_amount": round(amount, 2),
        "currency": currency,
        "transaction_type_dr_cr": direction,
        "transaction_mode_channel_bank": channel if not is_ppi else "",
        "cash_flag": is_cash,
        "transaction_type_ppi": ppi_txn_type,
        "transaction_mode_channel_ppi": ppi_channel,
        "transaction_status": status,
        "wallet_balance_before": w_before,
        "wallet_balance_after": w_after,
        "source_of_funds_wallet": random.choice(["Bank Transfer","UPI","Debit Card","Credit Card","Net Banking"]) if is_ppi else "",
        "load_instrument_type": random.choice(["Debit Card","Net Banking","UPI","Credit Card"]) if is_ppi else "",
        "load_source_account_card_details": f"XXXX-XXXX-{random.randint(1000,9999)}" if is_ppi else "",
        "beneficiary_wallet_id_vpa": generate_vpa(cp_name) if (is_ppi or channel == "UPI") else "",
        "merchant_id": m_id,
        "merchant_name": m_name,
        "merchant_category_code": mcc,
        "merchant_location": m_loc,
        "refund_chargeback_flag": random.choices(["Y","N"], weights=[0.02,0.98])[0] if not is_aml else "N",

        # -- Participant Information (SN 23-38) --
        "customer_account_number": acct_num,
        "account_wallet_status": acct_info.get("account_status", ""),
        "non_face_to_face_flag": cust.get("non_face_to_face_flag", ""),
        "pep_flag": cust.get("pep_flag", "N"),
        "hni_flag": cust.get("hni_flag", "N"),
        "minor_flag": cust.get("minor_flag", "N"),
        "customer_branch_ifsc_code": acct_info.get("customer_branch_ifsc", ""),
        "customer_cif_id": cif,
        "customer_cif_creation_date": cust.get("cif_creation_date", ""),
        "annual_income": cust.get("annual_income", ""),
        "counterparty_account_number": cp_acct,
        "counterparty_branch_ifsc_swift": cp_info.get("customer_branch_ifsc", generate_ifsc()),
        "customer_name": cust.get("customer_name", ""),
        "counterparty_name": cp_name,
        "sender_country_code": sender_cc,
        "receiver_country_code": receiver_cc,

        # -- Customer Profile Data (SN 39-74) --
        "customer_current_risk_score": cust.get("customer_risk_score", ""),
        "customer_type": cust.get("customer_type", ""),
        "customer_entity_type": cust.get("customer_entity_type", ""),
        "account_category": acct_info.get("account_category", ""),
        "account_type": acct_info.get("account_type", ""),
        "account_wallet_opening_date": acct_info.get("account_opening_date", ""),
        "customer_occupation_industry": cust.get("occupation_industry", ""),
        "vkyc_flag": cust.get("vkyc_flag", ""),
        "kyc_update_date": cust.get("kyc_update_date", ""),
        "account_wallet_inoperative_date": acct_info.get("inoperative_status_date", ""),
        "source_of_funds": cust.get("source_of_funds", ""),
        "tax_residency": cust.get("tax_residency", ""),
        "nationality": cust.get("nationality", ""),
        "citizenship": cust.get("citizenship", ""),
        "residency": cust.get("residency", ""),
        "date_of_incorporation": cust.get("date_of_incorporation", ""),
        "place_of_incorporation": cust.get("place_of_incorporation", ""),
        "beneficial_owner_types": cust.get("beneficial_owner_types", ""),
        "passive_nfe": cust.get("passive_nfe", ""),
        "address_registered_office": cust.get("address_registered_office", ""),
        "address_place_of_business": cust.get("address_place_of_business", ""),
        "address_beneficial_owners": cust.get("address_beneficial_owners", ""),
        "address_individual_customer": cust.get("address_individual", ""),
        "date_of_birth": cust.get("date_of_birth", ""),
        "father_spouse_name": cust.get("father_spouse_name", ""),
        "identification_proof_doc_no": cust.get("identification_doc_no", ""),
        "entity_identification_proof_doc_no": cust.get("entity_identification_doc_no", ""),
        "credit_summation_period": acct_info.get("credit_summation_period", 0),
        "debit_summation_period": acct_info.get("debit_summation_period", 0),
        "professional_experience_years": cust.get("professional_experience_years", ""),
        "cif_beneficial_owners": cust.get("cif_beneficial_owners", ""),
        "name_beneficial_owners": cust.get("name_beneficial_owners", ""),
        "mobile_number": cust.get("mobile_number", ""),
        "pan": cust.get("pan", ""),
        "aadhaar_number": cust.get("aadhaar_masked", ""),
        "email_id": cust.get("email_id", ""),

        # -- Wallet Account Data (SN 75-82) --
        "wallet_kyc_category": w_info.get("wallet_kyc_category", "") if is_ppi else "",
        "wallet_account_id": wallet_id if is_ppi else "",
        "escrow_account_linked": w_info.get("escrow_account_linked", "") if is_ppi else "",
        "transaction_limit_per_txn": w_info.get("per_txn_limit", "") if is_ppi else "",
        "daily_transaction_limit": w_info.get("daily_txn_limit", "") if is_ppi else "",
        "monthly_transaction_limit": w_info.get("monthly_txn_limit", "") if is_ppi else "",
        "annual_transaction_limit": w_info.get("annual_txn_limit", "") if is_ppi else "",
        "maximum_wallet_balance_limit": w_info.get("max_balance_limit", "") if is_ppi else "",

        # -- Device & Channel Data (SN 83-92) --
        "device_id_fingerprint": dev["device_id"] if dev else "",
        "ip_address": dev["ip_address"] if dev else "",
        "geo_location_city_country": f"{dev['geo_city']}, {dev['geo_country']}" if dev else "",
        "gps_coordinates_lat": dev["gps_lat"] if dev else "",
        "gps_coordinates_lon": dev["gps_lon"] if dev else "",
        "browser_app_information": dev["browser_app_info"] if dev else "",
        "session_id": session_id,
        "authentication_method": random.choice(CONFIG["auth_methods"]),
        "vpn_flag": dev["vpn_flag"] if dev else "N",
        "emulator_flag": dev["emulator_flag"] if dev else "N",
        "customer_address_lat": cust.get("address_lat", ""),
        "customer_address_lon": cust.get("address_lon", ""),

        # -- AML Labels --
        "is_aml": is_aml,
        "aml_typology": aml_typology,
        "typology_group_id": group_id,
    }

    # Assign unique txn ID
    tid = generate_txn_id()
    while tid in txn_id_set:
        tid = generate_txn_id()
    txn_id_set.add(tid)
    txn["transaction_id"] = tid

    return txn

print(f"Transaction builder ready. Output columns: {len(TRANSACTION_COLUMNS)}")



## 10 -- Compute Fraud Injection Plan


In [ ]:
# Estimate base transaction count
est_base_txns = 0
for acct in active_accounts_list:
    if acct["account_status"] == "Dormant":
        est_base_txns += 2
    else:
        lo, hi = CONFIG["num_transactions_per_account_range"]
        est_base_txns += (lo + hi) // 2

target_pct = CONFIG["target_fraud_pct"]
target_aml = int(est_base_txns * target_pct / (1.0 - target_pct))

weights = CONFIG["typology_weights"]
total_w = sum(weights.values())
norm_w = {k: v / total_w for k, v in weights.items()}
avg_per = CONFIG["avg_txns_per_scenario"]

SCENARIO_COUNTS = {}
print("=" * 75)
print("FRAUD INJECTION PLAN")
print("=" * 75)
print(f"  Estimated base txns:      {est_base_txns:>10,}")
print(f"  Target fraud %:           {target_pct*100:>9.1f}%")
print(f"  Target AML txns:          {target_aml:>10,}")
print()
print(f"  {'Typology':<35s} {'Weight':>7s} {'Target':>10s} {'Avg/Scen':>9s} {'Scenarios':>10s}")
print(f"  {'-'*73}")

for typ in weights:
    txn_target = int(target_aml * norm_w[typ])
    n_sc = max(1, math.ceil(txn_target / avg_per[typ]))
    SCENARIO_COUNTS[typ] = n_sc
    print(f"  {typ:<35s} {norm_w[typ]*100:>6.1f}% {txn_target:>10,} {avg_per[typ]:>9} {n_sc:>10,}")

print(f"  {'-'*73}")
print(f"  {'TOTAL':<35s} {'100.0%':>7s} {target_aml:>10,}")
print("=" * 75)



## 11 -- Generate All Transactions (Base + Embedded Typologies)
Base (clean) transactions and all 5 AML typologies are generated in a single pass.


In [ ]:
transactions = []
acct_credit_sums = defaultdict(float)
acct_debit_sums = defaultdict(float)

# ====================================================================
# PHASE 1: Generate base (clean) transactions
# ====================================================================
print("PHASE 1: Generating base transactions...")
for idx, acct in enumerate(active_accounts_list):
    if idx % 2000 == 0:
        print(f"  Account {idx+1:,}/{len(active_accounts_list):,}...")

    acct_num = acct["account_number"]
    cif = acct["customer_cif"]
    cust = cust_lookup.get(cif)
    if not cust:
        continue

    n_txns = random.randint(*CONFIG["num_transactions_per_account_range"])
    if acct["account_status"] == "Dormant":
        n_txns = random.randint(0, 5)

    income = cust["annual_income"] if isinstance(cust["annual_income"], int) else 500000
    monthly_budget = income / 12
    wallet_id = cif_to_wallet.get(cif, "")

    for _ in range(n_txns):
        txn_date_obj = random_date(CONFIG["date_range_start"], CONFIG["date_range_end"])
        txn_date = txn_date_obj.strftime("%d-%m-%Y")
        txn_time = unique_timestamp(acct_num, txn_date)

        direction = random.choices(["Dr","Cr"], weights=[0.55,0.45])[0]

        r = random.random()
        if r < 0.50:
            amount = round(random.uniform(50, 5000), 2)
        elif r < 0.80:
            amount = round(random.uniform(5000, 50000), 2)
        elif r < 0.95:
            amount = round(random.uniform(50000, 200000), 2)
        else:
            amount = round(random.uniform(200000, min(monthly_budget * 2, 5000000)), 2)

        channel = random.choice(CONFIG["bank_channels"])
        is_ppi = random.random() < 0.15 and wallet_id != ""
        ppi_channel = random.choice(CONFIG["ppi_channels"]) if is_ppi else ""
        ppi_txn_type = random.choice(["P2P","P2M","Bill Pay","Recharge","Cash Out"]) if is_ppi else ""

        cp_acct = random.choice(all_acct_numbers)
        while cp_acct == acct_num:
            cp_acct = random.choice(all_acct_numbers)

        txn = build_full_txn(acct_num, cp_acct, amount, txn_date, txn_time,
                             direction, channel, is_ppi=is_ppi,
                             ppi_txn_type=ppi_txn_type, ppi_channel=ppi_channel)
        transactions.append(txn)

        if direction == "Cr":
            acct_credit_sums[acct_num] += amount
        else:
            acct_debit_sums[acct_num] += amount

n_base = len(transactions)
print(f"  Base transactions generated: {n_base:,}")

# ====================================================================
# PHASE 2: Generate AML typology transactions
# ====================================================================
print("\nPHASE 2: Injecting AML typologies...")
aml_txns_generated = 0

def pick_n(n, exclude=None):
    exclude = set(exclude or [])
    pool = [a for a in active_only_numbers if a not in exclude]
    return random.sample(pool, min(n, len(pool)))

def random_date_str():
    start = datetime.strptime(CONFIG["date_range_start"], "%Y-%m-%d")
    end = datetime.strptime(CONFIG["date_range_end"], "%Y-%m-%d")
    d = start + timedelta(days=random.randint(0, (end-start).days))
    return d.strftime("%d-%m-%Y")

# -- T1: Structuring (Smurfing) --
print(f"  T1: Structuring ({SCENARIO_COUNTS['Structuring (Smurfing)']} scenarios)...")
for si in range(SCENARIO_COUNTS["Structuring (Smurfing)"]):
    gid = f"STRUCT_{si+1:05d}"
    target = random.choice(active_only_numbers)
    n_src = random.randint(3, 6)
    sources = pick_n(n_src, [target])
    if len(sources) < 3:
        continue
    ds = random_date_str()
    bh = random.randint(9, 16)
    for i, src in enumerate(sources):
        dep = random.uniform(8000, 9900)
        ts = unique_timestamp(src, ds, bh + (i % 6))
        txn = build_full_txn(src, src, dep, ds, ts, "Cr", "Branch Cash",
                             is_aml=1, aml_typology="Structuring (Smurfing)", group_id=gid)
        txn["cash_flag"] = "Y"
        transactions.append(txn)
        aml_txns_generated += 1
    nd = datetime.strptime(ds, "%d-%m-%Y") + timedelta(days=random.randint(1, 3))
    nds = nd.strftime("%d-%m-%Y")
    for src in sources:
        amt = random.uniform(7500, 9800)
        ts = unique_timestamp(src, nds, random.randint(10, 18))
        txn = build_full_txn(src, target, amt, nds, ts, "Dr",
                             random.choice(["NEFT","IMPS","UPI"]),
                             is_aml=1, aml_typology="Structuring (Smurfing)", group_id=gid)
        transactions.append(txn)
        aml_txns_generated += 1

# -- T2: Circular Transaction Loops --
print(f"  T2: Circular loops ({SCENARIO_COUNTS['Circular Transaction Loop']} scenarios)...")
for si in range(SCENARIO_COUNTS["Circular Transaction Loop"]):
    gid = f"CIRC_{si+1:05d}"
    ring_sz = random.randint(3, 5)
    ring = pick_n(ring_sz)
    if len(ring) < 3:
        continue
    base_amt = random.uniform(50000, 500000)
    ds = random_date_str()
    for hop in range(len(ring)):
        sender = ring[hop]
        receiver = ring[(hop + 1) % len(ring)]
        amt = base_amt * random.uniform(0.97, 1.0)
        hd = datetime.strptime(ds, "%d-%m-%Y") + timedelta(days=hop)
        hds = hd.strftime("%d-%m-%Y")
        ch = random.choice(["NEFT","RTGS","IMPS"])
        # Debit
        ts = unique_timestamp(sender, hds, random.randint(10, 17))
        txn = build_full_txn(sender, receiver, amt, hds, ts, "Dr", ch,
                             is_aml=1, aml_typology="Circular Transaction Loop", group_id=gid)
        transactions.append(txn)
        aml_txns_generated += 1
        # Credit
        ts2 = unique_timestamp(receiver, hds, random.randint(10, 17))
        txn2 = build_full_txn(receiver, sender, amt, hds, ts2, "Cr", ch,
                              is_aml=1, aml_typology="Circular Transaction Loop", group_id=gid)
        transactions.append(txn2)
        aml_txns_generated += 1

# -- T3: Funnel Account Networks --
print(f"  T3: Funnel networks ({SCENARIO_COUNTS['Funnel Account Network']} scenarios)...")
for si in range(SCENARIO_COUNTS["Funnel Account Network"]):
    gid = f"FUNNEL_{si+1:05d}"
    funnel = random.choice(active_only_numbers)
    n_feed = random.randint(15, 50)
    feeders = pick_n(n_feed, [funnel])
    dest_list = pick_n(1, [funnel] + feeders)
    if not dest_list or len(feeders) < 10:
        continue
    dest = dest_list[0]
    ds = random_date_str()
    total_in = 0
    for feeder in feeders:
        amt = random.uniform(5000, 30000)
        total_in += amt
        day_off = random.randint(0, 5)
        fd = datetime.strptime(ds, "%d-%m-%Y") + timedelta(days=day_off)
        fds = fd.strftime("%d-%m-%Y")
        ts = unique_timestamp(feeder, fds, random.randint(8, 20))
        txn = build_full_txn(feeder, funnel, amt, fds, ts, "Dr",
                             random.choice(["UPI","IMPS","NEFT"]),
                             is_aml=1, aml_typology="Funnel Account Network", group_id=gid)
        transactions.append(txn)
        aml_txns_generated += 1
    out_d = datetime.strptime(ds, "%d-%m-%Y") + timedelta(days=random.randint(6, 10))
    n_sp = random.randint(2, 3)
    rem = total_in * 0.95
    for s in range(n_sp):
        amt = rem * random.uniform(0.3, 0.5) if s < n_sp - 1 else rem
        rem -= amt
        od = out_d + timedelta(days=s)
        ods = od.strftime("%d-%m-%Y")
        ts = unique_timestamp(funnel, ods, random.randint(10, 16))
        txn = build_full_txn(funnel, dest, amt, ods, ts, "Dr",
                             "RTGS" if amt > 200000 else "NEFT",
                             is_aml=1, aml_typology="Funnel Account Network", group_id=gid)
        transactions.append(txn)
        aml_txns_generated += 1

# -- T4: Pass-Through Transit Hubs --
print(f"  T4: Pass-through ({SCENARIO_COUNTS['Pass-Through Transit Hub']} scenarios)...")
for si in range(SCENARIO_COUNTS["Pass-Through Transit Hub"]):
    gid = f"PASS_{si+1:05d}"
    transit = random.choice(active_only_numbers)
    src_list = pick_n(1, [transit])
    dst_list = pick_n(1, [transit] + src_list)
    if not src_list or not dst_list:
        continue
    src, dst = src_list[0], dst_list[0]
    amt_in = random.uniform(200000, 2000000)
    amt_out = amt_in * random.uniform(0.96, 0.99)
    ds = random_date_str()
    hour = random.randint(10, 17)
    # Inbound
    ts1 = unique_timestamp(transit, ds, hour)
    txn1 = build_full_txn(transit, src, amt_in, ds, ts1, "Cr",
                          random.choice(["RTGS","NEFT"]),
                          is_aml=1, aml_typology="Pass-Through Transit Hub", group_id=gid)
    txn1["counterparty_account_number"] = src
    transactions.append(txn1)
    aml_txns_generated += 1
    # Outbound
    ts2 = unique_timestamp(transit, ds, min(hour + random.choice([0,0,0,1]), 23))
    txn2 = build_full_txn(transit, dst, amt_out, ds, ts2, "Dr",
                          random.choice(["RTGS","NEFT","IMPS"]),
                          is_aml=1, aml_typology="Pass-Through Transit Hub", group_id=gid)
    transactions.append(txn2)
    aml_txns_generated += 1

# -- T5: Rapid Multi-Hop Layering --
print(f"  T5: Multi-hop layering ({SCENARIO_COUNTS['Rapid Multi-Hop Layering']} scenarios)...")
for si in range(SCENARIO_COUNTS["Rapid Multi-Hop Layering"]):
    gid = f"LAYER_{si+1:05d}"
    n_hops = random.randint(8, 10)
    chain = pick_n(n_hops + 1)
    if len(chain) < 9:
        continue
    base_amt = random.uniform(100000, 1000000)
    ds = random_date_str()
    start_h = random.randint(9, 14)
    for hop in range(len(chain) - 1):
        amt = base_amt * (0.99 ** hop) * random.uniform(0.98, 1.0)
        min_off = hop * random.randint(5, 30)
        hh = min(start_h + min_off // 60, 23)
        day_off = min_off // (24 * 60)
        hd = datetime.strptime(ds, "%d-%m-%Y") + timedelta(days=day_off)
        hds = hd.strftime("%d-%m-%Y")
        ts = unique_timestamp(chain[hop], hds, hh)
        txn = build_full_txn(chain[hop], chain[hop+1], amt, hds, ts, "Dr",
                             random.choice(["IMPS","NEFT","UPI","RTGS"]),
                             is_aml=1, aml_typology="Rapid Multi-Hop Layering", group_id=gid)
        transactions.append(txn)
        aml_txns_generated += 1

print(f"\n  AML transactions injected: {aml_txns_generated:,}")

# ====================================================================
# PHASE 3: Top-up if needed to hit target %
# ====================================================================
actual_total = len(transactions)
actual_aml = sum(1 for t in transactions if t["is_aml"] == 1)
actual_pct = actual_aml / actual_total

target_aml_exact = int(n_base * target_pct / (1.0 - target_pct))
deficit = target_aml_exact - actual_aml

if deficit > 0:
    print(f"\nPHASE 3: Top-up needed ({deficit:,} more AML txns)...")
    topup_round = 0
    while deficit > 0:
        topup_round += 1
        gid = f"TOPUP_{topup_round:05d}"
        choice = topup_round % 3
        if choice == 0:
            # pass-through
            tr = random.choice(active_only_numbers)
            s = pick_n(1, [tr])
            d = pick_n(1, [tr] + s)
            if not s or not d: continue
            ds = random_date_str(); h = random.randint(10, 17)
            ai = random.uniform(100000, 1500000); ao = ai * 0.97
            ts1 = unique_timestamp(tr, ds, h)
            t1 = build_full_txn(tr, s[0], ai, ds, ts1, "Cr", "RTGS",
                                is_aml=1, aml_typology="Pass-Through Transit Hub", group_id=gid)
            transactions.append(t1); deficit -= 1
            ts2 = unique_timestamp(tr, ds, min(h+1,23))
            t2 = build_full_txn(tr, d[0], ao, ds, ts2, "Dr", "NEFT",
                                is_aml=1, aml_typology="Pass-Through Transit Hub", group_id=gid)
            transactions.append(t2); deficit -= 1
        elif choice == 1:
            # structuring
            tgt = random.choice(active_only_numbers)
            srcs = pick_n(random.randint(3,5), [tgt])
            if len(srcs) < 3: continue
            ds = random_date_str()
            for src in srcs:
                ts = unique_timestamp(src, ds, random.randint(9,18))
                t = build_full_txn(src, src, random.uniform(8000,9900), ds, ts, "Cr", "Branch Cash",
                                   is_aml=1, aml_typology="Structuring (Smurfing)", group_id=gid)
                t["cash_flag"] = "Y"; transactions.append(t); deficit -= 1
            nd = datetime.strptime(ds, "%d-%m-%Y") + timedelta(days=1)
            nds = nd.strftime("%d-%m-%Y")
            for src in srcs:
                ts = unique_timestamp(src, nds, random.randint(10,18))
                t = build_full_txn(src, tgt, random.uniform(7500,9800), nds, ts, "Dr", "IMPS",
                                   is_aml=1, aml_typology="Structuring (Smurfing)", group_id=gid)
                transactions.append(t); deficit -= 1
        else:
            # layering
            ch = pick_n(random.randint(6,8))
            if len(ch) < 6: continue
            ba = random.uniform(80000, 600000); ds = random_date_str(); sh = random.randint(9,15)
            for hp in range(len(ch)-1):
                ts = unique_timestamp(ch[hp], ds, min(sh+hp,23))
                t = build_full_txn(ch[hp], ch[hp+1], ba*(0.99**hp), ds, ts, "Dr", "IMPS",
                                   is_aml=1, aml_typology="Rapid Multi-Hop Layering", group_id=gid)
                transactions.append(t); deficit -= 1
    print(f"  Top-up complete")
else:
    print("\nPHASE 3: No top-up needed")

# Final shuffle
random.shuffle(transactions)
total = len(transactions)
aml_count = sum(1 for t in transactions if t["is_aml"] == 1)
clean_count = total - aml_count

print(f"\n{'=' * 75}")
print(f"GENERATION COMPLETE")
print(f"{'=' * 75}")
print(f"  Clean transactions:  {clean_count:>10,}  ({clean_count/total*100:.2f}%)")
print(f"  AML transactions:    {aml_count:>10,}  ({aml_count/total*100:.2f}%)")
print(f"  Total transactions:  {total:>10,}")
print(f"  Columns per row:     {len(TRANSACTION_COLUMNS):>10}")
print(f"{'=' * 75}")



## 12 -- Timestamp Collision Check


In [ ]:
ts_check = defaultdict(list)
for t in transactions:
    ts_check[t["customer_account_number"]].append((t["datestamp"], t["timestamp"]))

collision_count = 0
for acct, ts_list in ts_check.items():
    c = Counter(ts_list)
    for k, v in c.items():
        if v > 1:
            collision_count += 1

if collision_count == 0:
    print("OK: No timestamp collisions detected")
else:
    print(f"Resolving {collision_count} timestamp collisions...")
    seen = defaultdict(set)
    for t in transactions:
        acct = t["customer_account_number"]
        key = (t["datestamp"], t["timestamp"])
        while key in seen[acct]:
            parts = t["timestamp"].split(":")
            s = int(parts[2]) + 1
            if s >= 60:
                s = 0; m = int(parts[1]) + 1
                if m >= 60:
                    m = 0; h = int(parts[0]) + 1
                    if h >= 24: h = 0
                    parts[0] = f"{h:02d}"
                parts[1] = f"{m:02d}"
            parts[2] = f"{s:02d}"
            t["timestamp"] = ":".join(parts)
            key = (t["datestamp"], t["timestamp"])
        seen[acct].add(key)
    print("  OK: All collisions resolved")



## 13 -- Update Account Credit/Debit Summations


In [ ]:
# Recompute from final transactions
final_credit = defaultdict(float)
final_debit = defaultdict(float)
for t in transactions:
    acct = t["customer_account_number"]
    amt = float(t["transaction_amount"])
    if t["transaction_type_dr_cr"] == "Cr":
        final_credit[acct] += amt
    else:
        final_debit[acct] += amt

# Update in transactions and accounts
for t in transactions:
    acct = t["customer_account_number"]
    t["credit_summation_period"] = round(final_credit[acct], 2)
    t["debit_summation_period"] = round(final_debit[acct], 2)

for a in accounts:
    a["credit_summation_period"] = round(final_credit[a["account_number"]], 2)
    a["debit_summation_period"] = round(final_debit[a["account_number"]], 2)

print("Account summations updated")



## 14 -- Export All Tables


In [ ]:
def save_csv(data, filename, fieldnames=None):
    if not data:
        print(f"  WARNING: {filename} empty, skipping"); return
    filepath = os.path.join(OUTPUT_DIR, filename)
    keys = fieldnames or list(data[0].keys())
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(data)
    print(f"  OK: {filename:40s} {len(data):>10,} rows x {len(keys):>3} cols")

print("Saving all datasets...")
save_csv(customers, "customers.csv")
save_csv(accounts, "accounts.csv")
save_csv(wallets, "wallets.csv")
save_csv(devices, "devices.csv")
save_csv(transactions, "transactions_final.csv", fieldnames=TRANSACTION_COLUMNS)

with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"  OK: {'config.json':40s}")

print(f"\nAll files saved to: {os.path.abspath(OUTPUT_DIR)}/")

print(f"\n-- File Sizes --")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fn)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  {fn:40s} {size_mb:>8.2f} MB")



## 15 -- Validation & Typology Distribution


In [ ]:
# Column count check
sample = transactions[0]
ordered_keys = TRANSACTION_COLUMNS
missing = [k for k in ordered_keys if k not in sample]
extra = [k for k in sample if k not in ordered_keys]
print(f"Column validation:")
print(f"  Expected:  {len(ordered_keys)}")
print(f"  In data:   {len(sample)}")
print(f"  Missing:   {missing if missing else 'None'}")
print(f"  Extra:     {extra if extra else 'None'}")

# Referential integrity
cif_cust = {c["customer_cif"] for c in customers}
cif_acct = {a["customer_cif"] for a in accounts}
cif_wall = {w["customer_cif"] for w in wallets}
print(f"\nReferential integrity:")
print(f"  Account CIFs in customers?  {cif_acct.issubset(cif_cust)}")
print(f"  Wallet CIFs in customers?   {cif_wall.issubset(cif_cust)}")

# Uniqueness
print(f"  Unique account numbers?     {len(set(a['account_number'] for a in accounts)) == len(accounts)}")
print(f"  Unique wallet IDs?          {len(set(w['wallet_id'] for w in wallets)) == len(wallets)}")

# Timestamp uniqueness
ts_per = defaultdict(list)
for t in transactions:
    ts_per[t["customer_account_number"]].append((t["datestamp"], t["timestamp"]))
dups = sum(sum(v-1 for v in Counter(tsl).values() if v > 1) for tsl in ts_per.values())
print(f"  Timestamp duplicates:       {dups} (should be 0)")

# Typology distribution
print(f"\n-- Typology Distribution --")
typ_counts = defaultdict(int)
typ_groups = defaultdict(set)
for t in transactions:
    typ = t.get("aml_typology", "")
    if typ:
        typ_counts[typ] += 1
        typ_groups[typ].add(t.get("typology_group_id", ""))

total_aml = sum(typ_counts.values())
for typ in sorted(typ_counts):
    cnt = typ_counts[typ]
    grps = len(typ_groups[typ])
    print(f"  {typ:<35s} {cnt:>7,} txns ({cnt/total_aml*100:>5.1f}% of AML) [{grps:>5} scenarios]")

print(f"\n  {'TOTAL AML':<35s} {total_aml:>7,} ({total_aml/len(transactions)*100:.2f}% of total)")
print(f"  {'TOTAL CLEAN':<35s} {len(transactions)-total_aml:>7,} ({(len(transactions)-total_aml)/len(transactions)*100:.2f}% of total)")
print(f"  {'GRAND TOTAL':<35s} {len(transactions):>7,}")

print(f"\n{'=' * 60}")
print(f"PIPELINE COMPLETE - All data ready for model training!")
print(f"{'=' * 60}")

